# Exercice 08 : Nettoyer et justifier

In [2]:
import ssl
from pathlib import Path
import seaborn as sns

ssl._create_default_https_context = ssl._create_unverified_context
df = sns.load_dataset("titanic")
print(df.shape)
print(df.head())

(891, 15)
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


## 1. Valeurs manquantes

In [3]:
print(df.isna().sum())
print(df.isna().mean().round(3))

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
survived       0.000
pclass         0.000
sex            0.000
age            0.199
sibsp          0.000
parch          0.000
fare           0.000
embarked       0.002
class          0.000
who            0.000
adult_male     0.000
deck           0.772
embark_town    0.002
alive          0.000
alone          0.000
dtype: float64


## 2. Doublons

On les examine avant toute imputation : remplir `age` avec une médiane créerait de faux doublons.

In [4]:
nb_doublons = df.duplicated().sum()
print(f"Doublons : {nb_doublons}")
doublons = df[df.duplicated(keep=False)]
print(doublons.sort_values(by=["pclass", "sex", "age", "fare"]).head(10))

Doublons : 107
     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
369         1       1  female  24.0      0      0  69.3000        C   First   
641         1       1  female  24.0      0      0  69.3000        C   First   
64          0       1    male   NaN      0      0  27.7208        C   First   
295         0       1    male   NaN      0      0  27.7208        C   First   
443         1       2  female  28.0      0      0  13.0000        S  Second   
635         1       2  female  28.0      0      0  13.0000        S  Second   
53          1       2  female  29.0      1      0  26.0000        S  Second   
133         1       2  female  29.0      1      0  26.0000        S  Second   
458         1       2  female  50.0      0      0  10.5000        S  Second   
526         1       2  female  50.0      0      0  10.5000        S  Second   

       who  adult_male deck  embark_town alive  alone  
369  woman       False    B    Cherbourg   yes   True  
641

| Quoi | Pourquoi | Lignes |
|------|----------|--------|
| doublons conservés | pas d'identifiant passager : deux lignes identiques peuvent être deux personnes réelles (même classe, même tarif, même port) | 0 |

## 3. Traitement des manquants

In [5]:
nb_ages_manquants = df["age"].isna().sum()
df["age"] = df["age"].fillna(
    df.groupby(["pclass", "sex"])["age"].transform("median")
)
print(f"Âges imputés : {nb_ages_manquants}")

Âges imputés : 177


| Quoi | Pourquoi | Lignes |
|------|----------|--------|
| age imputé | médiane de la classe et du sexe : l'âge varie selon ces deux catégories, la variable est trop utile pour être supprimée | 177 |

In [6]:
nb_sans_port = df["embark_town"].isna().sum()
df = df.dropna(subset=["embark_town"])
print(f"Passagers sans port supprimés : {nb_sans_port}")

Passagers sans port supprimés : 2


| Quoi | Pourquoi | Lignes |
|------|----------|--------|
| lignes sans embark_town supprimées | très peu de lignes concernées (0,2 %), impossible de deviner le port | 2 |

In [7]:
part_deck_manquant = df["deck"].isna().mean()
df = df.drop(columns=["deck"])
print(f"deck manquant : {part_deck_manquant:.0%}")

deck manquant : 77%


| Quoi | Pourquoi | Lignes |
|------|----------|--------|
| deck supprimée | 77 % de manquants : trop peu de valeurs pour servir | 889 |

## 4. Colonnes redondantes et types

In [8]:
df = df.drop(columns=["alive", "pclass", "embarked", "who", "adult_male"])
df["survived"] = df["survived"].astype(bool)
print(df.dtypes)

survived           bool
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
class          category
embark_town         str
alone              bool
dtype: object


| Quoi | Pourquoi | Lignes |
|------|----------|--------|
| alive supprimée | même information que survived (yes/no) | 889 |
| pclass supprimée | même information que class (1/2/3 et First/Second/Third) | 889 |
| embarked supprimée | même information que embark_town (S/C/Q) | 889 |
| who et adult_male supprimées | se déduisent de sex et age | 889 |
| survived convertie en booléen | 0/1 signifie oui/non ; la moyenne donne toujours le taux de survie | 889 |
| age gardé en décimal | 202 âges non entiers (bébés de quelques mois) : un entier les ramènerait à 0 | 0 |

## 5. Valeurs aberrantes

In [9]:
print(df["fare"].describe())

q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
ecart = q3 - q1
borne_haute = q3 + 1.5 * ecart

tres_chers = df[df["fare"] > borne_haute]
print(f"Borne haute : {borne_haute:.2f}")
print(f"Billets au-dessus : {len(tres_chers)}")
print(tres_chers["class"].value_counts())

billets_gratuits = df[df["fare"] == 0]
print(f"Billets à 0 : {len(billets_gratuits)}")

count    889.000000
mean      32.096681
std       49.697504
min        0.000000
25%        7.895800
50%       14.454200
75%       31.000000
max      512.329200
Name: fare, dtype: float64
Borne haute : 65.66
Billets au-dessus : 114
class
First     102
Third       7
Second      5
Name: count, dtype: int64
Billets à 0 : 15


| Quoi | Pourquoi | Lignes |
|------|----------|--------|
| billets au-dessus de la borne haute conservés | presque tous en 1re classe : valeurs rares mais réelles, pas des erreurs de saisie. Aberrant ne veut pas dire faux | 0 || billets à 0 conservés | répartis dans les 3 classes, probablement des invitations ou de l'équipage : plausible, pas une erreur évidente | 0 |


## 6. Enregistrement

In [10]:
print(df.isna().sum())
print(df.shape)

Path("data").mkdir(exist_ok=True)
df.to_csv("data/titanic_propre.csv", index=False)

survived       0
sex            0
age            0
sibsp          0
parch          0
fare           0
class          0
embark_town    0
alone          0
dtype: int64
(889, 9)
